# Session 12.2: Feature Engineering & Sklearn Pipelines

<table cellpadding="10" cellspacing="0" border="0" width="100%"><tr>
<td bgcolor="#ED1C24" width="4"></td>
<td bgcolor="#fce4ec">
<font color="#c62828"><b>Program:</b></font> Vishlesan i-Hub IIT Patna x Masai School -- AIM (AI & Machine Learning)<br>
<font color="#c62828"><b>Session ID:</b></font> 12.2 | <b>Week:</b> 12 | <b>Module:</b> Module 1 — Foundations of AI and Data Science<br>
<font color="#c62828"><b>Prerequisites:</b></font> Sessions 9.1, 9.2, 10.1, 11.1–11.2, 12.1<br>
<font color="#c62828"><b>Estimated completion time:</b></font> 90–120 minutes (self-paced)
</td>
</tr></table>

---

Vishlesan i-Hub IIT Patna × Masai School

## Learning Objectives

By the end of this notebook you will be able to:

1. **Diagnose target leakage** by asking *"Could this feature exist before the prediction time?"* — demonstrated on the Bank Marketing `duration` column.
2. **Apply 6 feature creation patterns** — extract, transform, combine, bin, sentinel handling, frequency encoding.
3. **Build a `ColumnTransformer`** that routes numeric to a scaler and categorical to `OneHotEncoder(handle_unknown='ignore')`.
4. **Compose `ColumnTransformer` + estimator into a `Pipeline`** with `set_output(transform='pandas')` and `get_feature_names_out()`.
5. **Generate interaction features** with `PolynomialFeatures(degree=2, interaction_only=True)` and explain the combinatorial expansion.
6. **Cross-validate the full Pipeline** with `StratifiedKFold` + `scoring='average_precision'` — leakage-free by construction.
7. **Serialize the fitted Pipeline** with `joblib.dump` and verify a round-trip in a fresh environment.
8. **Articulate the security warning** — `joblib.load` executes arbitrary code.
9. **Reach F1 ≥ 0.50** on the Bank Marketing subscription class with the full ColumnTransformer pipeline.


## Prerequisites

- **Session 9.1** — sklearn `Pipeline` API; `fit / predict`. Today we upgrade with `ColumnTransformer` and serialization.
- **Session 9.2** — encoding, EDA habits, missing-value strategies.
- **Session 10.1** — KNN distance metrics — *why* scaling matters before KNN.
- **Sessions 11.1–11.2** — Stratified train/test split discipline.
- **Session 12.1** — `class_weight='balanced'`, PR-AUC, leakage anti-pattern. **All carry forward.** Bank Marketing's ~11% positive class is moderate enough that `class_weight='balanced'` alone is sufficient — no SMOTE today.


## Setup & Imports

In [1]:
# All libraries pre-installed in Colab; pin minimums for reproducibility
!pip install -q -U "scikit-learn>=1.4" "plotly>=5.20" "joblib>=1.4"
!pip install --upgrade gdown


In [2]:
!gdown 1g4TCUoLtzGd79WYO-bXqx8IkZNvhpeEp

Downloading...
From: https://drive.google.com/uc?id=1g4TCUoLtzGd79WYO-bXqx8IkZNvhpeEp
To: d:\Languages\Python\Collab-Notebooks\supervised_learning\bank-additional-full.csv

  0%|          | 0.00/5.83M [00:00<?, ?B/s]
  9%|▉         | 524k/5.83M [00:00<00:01, 3.61MB/s]
 36%|███▌      | 2.10M/5.83M [00:00<00:00, 8.99MB/s]
 81%|████████  | 4.72M/5.83M [00:00<00:00, 14.8MB/s]
100%|██████████| 5.83M/5.83M [00:00<00:00, 13.8MB/s]


In [3]:
import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "colab"

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    average_precision_score, roc_auc_score,
    classification_report, confusion_matrix,
)
import joblib
import sklearn

RNG = 42
MASAI_RED = "#ED1C24"
print("Setup complete. sklearn version:", sklearn.__version__)


Setup complete. sklearn version: 1.9.0


## The `box()` helper for styled callouts

We use this throughout the notebook for definitions, tips, warnings, math, and outputs.

In [4]:
from IPython.display import HTML, display

_BOX_STYLES = {
    "definition": ("#448aff", "#e3f2fd", "#1565c0"),
    "tip":        ("#00c853", "#e8f5e9", "#2e7d32"),
    "warning":    ("#ff9100", "#fff3e0", "#e65100"),
    "danger":     ("#ff1744", "#fce4ec", "#c62828"),
    "math":       ("#7c4dff", "#ede7f6", "#4527a0"),
    "output":     ("#00b8d4", "#e0f7fa", "#006064"),
    "industry":   ("#009688", "#e0f2f1", "#004d40"),
}

def box(kind, title, content):
    border, bg, title_clr = _BOX_STYLES[kind]
    display(HTML(f"""
    <div style="margin:12px 0; color: gray; padding:12px 16px; border-left:4px solid {border};
                background-color:{bg}; border-radius:4px;">
    <strong style="color:{title_clr};">{title}</strong><br>{content}
    </div>"""))


## Interactive precompute zone

Several charts later in this notebook are interactive — click a button or drag a slider, the model's behavior updates. Plotly's animation engine cannot re-fit sklearn models on the fly, so we **fit every preprocessing variant once here** on the real Bank Marketing dataset and cache the results. After this cell, every interactive plot reads from the cache. **No synthetic data anywhere in this notebook \u2014 every visualization is on the real `bank-additional-full.csv`.**


In [5]:
df_pre = pd.read_csv("bank-additional-full.csv", sep=";")
y_pre = (df_pre["y"] == "yes").astype(int)
X_pre = df_pre.drop(columns=["y", "duration"])  # drop the leaky column up-front
X_tr_pre, X_te_pre, y_tr_pre, y_te_pre = train_test_split(
    X_pre, y_pre, stratify=y_pre, test_size=0.2, random_state=RNG,
)
num_cols_pre = X_pre.select_dtypes(include="number").columns.tolist()
cat_cols_pre = X_pre.select_dtypes(include="object").columns.tolist()
print(f"Bank Marketing (no duration): train={len(X_tr_pre):,}, test={len(X_te_pre):,}")
print(f"Numeric: {len(num_cols_pre)}, Categorical: {len(cat_cols_pre)}")


Bank Marketing (no duration): train=32,950, test=8,238
Numeric: 9, Categorical: 10


In [6]:
PREP_STRATEGIES = [
    "Raw numeric (no preprocessing)",
    "Numeric scaling only",
    "Full ColumnTransformer",
    "+ Polynomial interactions",
    "+ class_weight='balanced'",
]

def _fit_prep(strategy):
    """Fit a preprocessing variant on Bank Marketing; return metrics + probas + cm."""
    if strategy == "Raw numeric (no preprocessing)":
        clf = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RNG)
        clf.fit(X_tr_pre[num_cols_pre], y_tr_pre)
        proba = clf.predict_proba(X_te_pre[num_cols_pre])[:, 1]
    elif strategy == "Numeric scaling only":
        pipe = Pipeline([("sc", StandardScaler()),
                         ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RNG))])
        pipe.fit(X_tr_pre[num_cols_pre], y_tr_pre)
        proba = pipe.predict_proba(X_te_pre[num_cols_pre])[:, 1]
    elif strategy == "Full ColumnTransformer":
        pipe = Pipeline([
            ("prep", ColumnTransformer([
                ("num", StandardScaler(), num_cols_pre),
                ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols_pre),
            ])),
            ("clf", LogisticRegression(max_iter=2000, random_state=RNG))])
        pipe.fit(X_tr_pre, y_tr_pre)
        proba = pipe.predict_proba(X_te_pre)[:, 1]
    elif strategy == "+ Polynomial interactions":
        # Apply polynomial only to a subset of numeric features (keeps memory sane)
        poly_num = num_cols_pre[:5]  # first 5 numeric features
        other_num = [c for c in num_cols_pre if c not in poly_num]
        pipe = Pipeline([
            ("prep", ColumnTransformer([
                ("poly_num", Pipeline([("sc", StandardScaler()),
                                       ("poly", PolynomialFeatures(degree=2, interaction_only=True, include_bias=False))]), poly_num),
                ("other_num", StandardScaler(), other_num),
                ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols_pre),
            ])),
            ("clf", LogisticRegression(max_iter=2000, C=0.5, random_state=RNG))])
        pipe.fit(X_tr_pre, y_tr_pre)
        proba = pipe.predict_proba(X_te_pre)[:, 1]
    elif strategy == "+ class_weight='balanced'":
        pipe = Pipeline([
            ("prep", ColumnTransformer([
                ("num", StandardScaler(), num_cols_pre),
                ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols_pre),
            ])),
            ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RNG))])
        pipe.fit(X_tr_pre, y_tr_pre)
        proba = pipe.predict_proba(X_te_pre)[:, 1]
    y_pred = (proba >= 0.5).astype(int)
    cm = confusion_matrix(y_te_pre, y_pred)
    return dict(
        cm=cm, proba=proba,
        F1=f1_score(y_te_pre, y_pred, zero_division=0),
        Precision=precision_score(y_te_pre, y_pred, zero_division=0),
        Recall=recall_score(y_te_pre, y_pred),
        AP=average_precision_score(y_te_pre, proba),
        ROC_AUC=roc_auc_score(y_te_pre, proba),
    )

PREP_CACHE = {s: _fit_prep(s) for s in PREP_STRATEGIES}
for s in PREP_STRATEGIES:
    r = PREP_CACHE[s]
    print(f"  {s:36s}  F1={r['F1']:.3f}  PR-AUC={r['AP']:.3f}  ROC-AUC={r['ROC_AUC']:.3f}")


  Raw numeric (no preprocessing)        F1=0.389  PR-AUC=0.443  ROC-AUC=0.776
  Numeric scaling only                  F1=0.389  PR-AUC=0.443  ROC-AUC=0.776
  Full ColumnTransformer                F1=0.332  PR-AUC=0.465  ROC-AUC=0.801
  + Polynomial interactions             F1=0.337  PR-AUC=0.468  ROC-AUC=0.802
  + class_weight='balanced'             F1=0.469  PR-AUC=0.460  ROC-AUC=0.801


## The Business Hook — Three Stories, One Lesson

**Zillow iBuying — $881M loss (November 2021).**

Zillow's algorithmic home-valuation pipeline trained on pre-pandemic data could not adapt to post-2020 market volatility. Missing features around local supply ratios + an ill-fated initiative that *prevented humans from overriding the model* led to 7,000 over-priced home purchases and a writedown of **$881 million**. Zillow shut down iBuying in November 2021. *Lesson: feature engineering is the highest-leverage decision in any ML project.*

**Excel gene-name corruption — 20% of biology papers (Ziemann 2016 → HUGO 2020).** Three biologists found in *Genome Biology* that ~1 in 5 genetics papers had silently corrupted gene names because Excel auto-converted symbols like `MARCH1`, `SEPT2`, `SEPT9` into dates. The HUGO Gene Nomenclature Committee responded in **August 2020 by formally renaming 27 genes** (e.g., `MARCH1` → `MARCHF1`) just to defeat Excel. *Lesson: silent feature corruption is the worst kind. Reproducible Pipelines catch it.*

**The `duration` leakage trap living *inside* today's dataset.** UCI Bank Marketing has a `duration` column — the call length in seconds. Using it as a predictor gives F1 ≈ 0.87. **But `duration` is only known *after* the call ends.** It is target leakage. The dataset documentation literally warns about this. We will demonstrate the leakage and the fix in Section 1.


---

## The Business Case & The Dataset

Before we touch any code, let us be explicit about *what* we are predicting and *for whom*. The next 100 minutes are easier to follow when the answer to 'why does this matter?' is in your head.


In [7]:
box("industry", "Portuguese Bank Telemarketing — The Business Case",
    "<b>Setting.</b> A Portuguese retail bank, between May 2008 and November 2010 — the height of the global financial crisis. "
    "The bank wants to grow its base of <i>term-deposit</i> customers (people who lock money with the bank for a fixed period in exchange for guaranteed interest).<br><br>"
    "<b>How they do it today.</b> Outbound telemarketing. Agents call customers from a list and try to convince them to subscribe. "
    "Each call costs agent time, training, and infrastructure.<br><br>"
    "<b>The pain.</b> Across the 2.5-year window in this dataset, only <b>~11.3% of calls succeed</b> — roughly 1 in 9. "
    "The other 88.7% are time the bank could have spent on more promising leads. Calling everyone is wasteful.<br><br>"
    "<b>The ML question.</b> <i>'Given everything we know about a customer <b>before</b> we pick up the phone — their age, job, education, "
    "prior contact history, and current macroeconomic conditions — can we predict who is most likely to say yes?'</i> "
    "If we can, the bank prioritises calls and improves conversion <i>without</i> adding agents. "
    "This is binary classification: target <code>y</code> takes values <code>yes</code> / <code>no</code>.<br><br>"
    "<b>Why it matters beyond banking.</b> The same logic underpins every sales-funnel prioritisation problem in industry — "
    "insurance lead scoring, B2B SaaS demo qualification, NBFC loan cross-sell, e-commerce re-targeting. "
    "Get good at this dataset and you have a template for all of them.")

### The 21 columns, grouped by type

**10 numeric features** — about the customer, the campaign, and the macroeconomic backdrop:

| Column | What it is |
|---|---|
| `age` | Customer's age in years |
| `duration` | **Call duration in seconds.** ⚠️ Target leakage — known only *after* the call. F1 = 0.87 with it (a lie); F1 ≈ 0.40 without (the truth). Section 1 demonstrates this live. |
| `campaign` | Number of contacts performed during *this* campaign with this customer |
| `pdays` | Days since the last contact in a previous campaign. ⚠️ **`999` is a sentinel meaning 'never contacted before'** — affects ~96% of rows, NOT a real count |
| `previous` | Number of contacts in *previous* campaigns with this customer |
| `emp.var.rate` | Employment variation rate — quarterly macroeconomic indicator |
| `cons.price.idx` | Consumer price index — monthly |
| `cons.conf.idx` | Consumer confidence index — monthly |
| `euribor3m` | 3-month Euro Interbank Offered Rate — daily; the bank's wholesale funding cost |
| `nr.employed` | Number of employees in the economy — quarterly |

**10 categorical features** — about the customer's profile and how the contact was made:

| Column | Sample categories | What it is |
|---|---|---|
| `job` | admin, blue-collar, technician, services, ... (12 in total) | Customer's occupation |
| `marital` | married / single / divorced / unknown | Marital status |
| `education` | basic.4y / high.school / university.degree / ... | Highest education completed |
| `default` | yes / no / unknown | Has the customer defaulted on credit before? |
| `housing` | yes / no / unknown | Has a housing loan? |
| `loan` | yes / no / unknown | Has any personal loan? |
| `contact` | telephone / cellular | How was this contact made? |
| `month` | jan, feb, ..., dec | Month of last contact (cyclical — `month=12` is *next to* `month=1`) |
| `day_of_week` | mon, tue, wed, thu, fri | Day of last contact (cyclical) |
| `poutcome` | success / failure / nonexistent | Outcome of the *previous* marketing campaign with this customer |

**1 target column:**

| Column | Values | Note |
|---|---|---|
| `y` | `yes` / `no` (string) | Did the customer subscribe? **~11.3% positive class** — moderately imbalanced; we will use `class_weight='balanced'` and PR-AUC, not accuracy |

**Total:** 41,188 rows × 21 columns. Source: UCI Machine Learning Repository, dataset 222 — *Bank Marketing*. File: `bank-additional-full.csv` (~5.8 MB).


In [8]:
box("warning", "Three Quirks To Notice Before You Type pd.read_csv",
    "<b>1. The separator is <code>;</code> (semicolon), not <code>,</code>.</b> A UCI quirk. "
    "You must pass <code>sep=';'</code> to <code>pd.read_csv</code>. Forgetting this gives you a single-column DataFrame "
    "with the entire row stuffed into one cell — silent failure mode, easy to miss.<br><br>"
    "<b>2. <code>pdays = 999</code> is a sentinel, not a count.</b> About 96% of rows have <code>pdays = 999</code>, "
    "meaning 'this customer was never contacted before'. It is a special-value placeholder, NOT a real number of days. "
    "If you run <code>StandardScaler</code> on this column without splitting out the sentinel, the mean and std are dominated "
    "by the 999s and the genuine numeric information for the 4% who <i>were</i> previously contacted gets crushed. "
    "Section 1 covers the split-into-flag-plus-clean fix (sentinel handling — pattern 5 of 6).<br><br>"
    "<b>3. <code>duration</code> is the most expensive feature in the dataset.</b> Including it as a predictor gives "
    "F1 ≈ 0.87. Excluding it drops F1 to ~0.40. The 0.87 number is a lie — <code>duration</code> is the call's length "
    "in seconds, and you only know it <i>after</i> the call has ended, by which point the answer is also known. "
    "This is target leakage and it is the most important moment in this entire session. Section 1.1–1.3 reproduces both numbers "
    "so you see the trap with your own eyes.")

---

## Section 1 — Feature Creation Strategies

Diagnose first; engineer second.

### 1.1 — Load the Bank Marketing dataset

**Note:** The CSV uses `;` (semicolon) as separator, not comma — a UCI quirk. Always verify your separator.


In [9]:
df = pd.read_csv("bank-additional-full.csv", sep=";")
print("Shape:", df.shape)
print("\nFirst 3 rows:")
df.head(3)


Shape: (41188, 21)

First 3 rows:


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [11]:
y = (df["y"] == "yes").astype(int)
X = df.drop(columns=["y"])

n_pos, n_neg = int(y.sum()), int((y == 0).sum())
rate = y.mean()
print(f"Positive (subscribed)    : {n_pos:,}  ({rate*100:.2f}%)")
print(f"Negative (did not)       : {n_neg:,}  ({(1-rate)*100:.2f}%)")
print(f"Imbalance ratio          : {n_neg/n_pos:.1f} : 1")


Positive (subscribed)    : 4,640  (11.27%)
Negative (did not)       : 36,548  (88.73%)
Imbalance ratio          : 7.9 : 1


In [12]:
num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = X.select_dtypes(include="object").columns.tolist()
print(f"Numeric columns ({len(num_cols)}):    {num_cols}")
print(f"Categorical columns ({len(cat_cols)}): {cat_cols}")


Numeric columns (10):    ['age', 'duration', 'campaign', 'pdays', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']
Categorical columns (10): ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome']


#### What an 11.27% positive class looks like

Less extreme than the 0.17% fraud problem from 12.1, but still imbalanced. The visual sets your expectations: `class_weight='balanced'` should be enough, no SMOTE needed.


In [13]:
fig = go.Figure(data=[
    go.Bar(x=[n_neg], y=['Customers'], orientation='h', name='Did not subscribe',
           marker=dict(color='#888'),
           hovertemplate=f'Did not subscribe: {n_neg:,} ({(1-rate)*100:.2f}%)<extra></extra>'),
    go.Bar(x=[n_pos], y=['Customers'], orientation='h', name='Subscribed',
           marker=dict(color=MASAI_RED),
           hovertemplate=f'Subscribed: {n_pos:,} ({rate*100:.2f}%)<extra></extra>'),
])
fig.update_layout(barmode='stack',
    title=f'{len(df):,} customers — {n_pos:,} subscribed ({rate*100:.2f}%)',
    template='plotly_white', height=180, margin=dict(l=80, r=20, t=60, b=40),
    legend=dict(orientation='h', y=-0.4))
fig.show()


### 1.2 — The `duration` leakage trap (live demo)

Build a Pipeline that includes ALL features (including `duration`), train, evaluate. F1 will look great. It is a lie.


**Why this matters.** A *leaky* feature is one whose value would not be available at prediction time. `duration` (the call's length in seconds) only exists *after* the call ends — by which point the answer (`y` = yes/no) is also known. A model trained with `duration` is essentially using the future to predict the future. The F1 will look great in the notebook and the model will fail in production.

**Our options for handling a leaky feature:**

| Option | What it does | Trade-off |
|---|---|---|
| **Drop the feature** | Remove from `X` entirely | Loses predictive power *that was a lie anyway*; gives an honest baseline. **The right default.** |
| **Use it but caveat** | Keep for benchmark scoring only, never for deployment | Easy to forget the caveat downstream; risky in shared notebooks |
| **Substitute a non-leaky proxy** | E.g., predict expected `duration` from pre-call features, use that | Adds a second model + complexity; only worth it if the proxy itself is strong |
| **Reframe the prediction time** | Predict mid-call instead of pre-call | Changes the business problem; rarely the right move |

We will use **drop** below. First we *demonstrate the leak* (1.2), then *drop and re-evaluate* (1.3). The reasoning for the choice is in the cell after the leakage-tax visualization.


In [14]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, stratify=y, test_size=0.2, random_state=RNG)

leaky_pipe = Pipeline([
    ("preprocess", ColumnTransformer([
        ("num", StandardScaler(),                                num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
    ])),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RNG)),
])
leaky_pipe.fit(X_tr, y_tr)
y_pred_leaky = leaky_pipe.predict(X_te)
y_proba_leaky = leaky_pipe.predict_proba(X_te)[:, 1]

leaky_f1 = f1_score(y_te, y_pred_leaky)
leaky_ap = average_precision_score(y_te, y_proba_leaky)
print(f"WITH duration: F1 = {leaky_f1:.3f}, PR-AUC = {leaky_ap:.3f}")


WITH duration: F1 = 0.604, PR-AUC = 0.622


**That F1 looks suspiciously good.** Let's plot the `duration` distribution split by `y` to see the smoking gun.

In [15]:
fig = px.histogram(
    df, x="duration", color="y", nbins=60, opacity=0.7, barmode="overlay",
    color_discrete_map={"no": "#888", "yes": MASAI_RED},
    title="`duration` distribution by outcome — the bimodal pattern is the smoking gun",
)
fig.update_layout(template="plotly_white", height=380)
fig.show()


In [16]:
median_no = df.loc[df["y"] == "no", "duration"].median()
median_yes = df.loc[df["y"] == "yes", "duration"].median()
box("output", "What this chart shows",
    f"Median <code>duration</code> when y='no' = <b>{median_no:.0f} seconds</b>. "
    f"Median <code>duration</code> when y='yes' = <b>{median_yes:.0f} seconds</b> — almost <b>{median_yes/median_no:.1f}× longer</b>. "
    "This isn't because long calls cause subscriptions — it's because the bank kept customers on the line until they agreed (or hung up quickly when they didn't). "
    "<b><code>duration</code> encodes the answer.</b>")


In [17]:
box("danger", "The single most important question in feature engineering",
    "<b><i>'Could this feature exist <u>before</u> the prediction time?'</i></b><br><br>"
    "If the answer is no, the feature is target leakage — it tells the model the answer in disguise. <b>Drop it.</b><br><br>"
    "The Bank Marketing documentation states explicitly: <i>\"this attribute highly affects the output target (e.g., if duration=0 then y='no'). Yet, the duration is not known before a call is performed. Also, after the end of the call y is obviously known. Thus, this input should only be included for benchmark purposes and should be discarded if the intention is to have a realistic predictive model.\"</i>")


### 1.3 — The honest baseline (drop `duration`)

In [18]:
X_honest = X.drop(columns=["duration"])
X_tr_h, X_te_h, y_tr_h, y_te_h = train_test_split(
    X_honest, y, stratify=y, test_size=0.2, random_state=RNG,
)
num_cols_h = X_honest.select_dtypes(include="number").columns.tolist()
cat_cols_h = X_honest.select_dtypes(include="object").columns.tolist()

honest_pipe = Pipeline([
    ("preprocess", ColumnTransformer([
        ("num", StandardScaler(),                                num_cols_h),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols_h),
    ])),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RNG)),
])
honest_pipe.fit(X_tr_h, y_tr_h)
honest_f1 = f1_score(y_te_h, honest_pipe.predict(X_te_h))
honest_ap = average_precision_score(y_te_h, honest_pipe.predict_proba(X_te_h)[:, 1])
print(f"WITHOUT duration: F1 = {honest_f1:.3f}, PR-AUC = {honest_ap:.3f}")
print(f"\n  Drop in F1     : {leaky_f1:.3f} → {honest_f1:.3f}  ({(honest_f1-leaky_f1)/leaky_f1*100:+.1f}%)")
print(f"  Drop in PR-AUC : {leaky_ap:.3f} → {honest_ap:.3f}")


WITHOUT duration: F1 = 0.469, PR-AUC = 0.460

  Drop in F1     : 0.604 → 0.469  (-22.3%)
  Drop in PR-AUC : 0.622 → 0.460


In [19]:
box("output", "What just happened",
    f"Dropping <code>duration</code> made F1 fall from <b>{leaky_f1:.3f}</b> to <b>{honest_f1:.3f}</b>. "
    "<b>This is exactly correct.</b> The first number was a lie; the second number is reality. "
    "Building features on top of a lie sends you down the Zillow path. Building features on top of reality is the job.")


#### The leakage tax — visualized

In [20]:
fig = go.Figure(data=[
    go.Bar(x=['WITH duration<br>(leaky)', 'WITHOUT duration<br>(honest)'],
           y=[leaky_f1, honest_f1],
           text=[f'F1={leaky_f1:.3f}', f'F1={honest_f1:.3f}'],
           textposition='outside',
           marker=dict(color=['#ff1744', '#2E7D32']),
           hovertemplate='%{x}<br>F1=%{y:.3f}<extra></extra>'),
])
fig.add_annotation(
    x=0.5, y=max(leaky_f1, honest_f1) * 1.05, xref='x', yref='y',
    text=f"\u0394 F1 = {leaky_f1 - honest_f1:.3f}<br>(the leakage tax)",
    showarrow=True, arrowhead=2, ax=0, ay=-30,
    bgcolor='#fff3e0', bordercolor='#e65100', borderwidth=1, borderpad=6,
    font=dict(family='monospace', size=12),
)
fig.update_layout(
    title='Same model. Same data. Drop one leaky feature, lose 0.13 F1 \u2014 because the 0.87 was a lie.',
    template='plotly_white', height=380, showlegend=False,
    yaxis=dict(range=[0, max(leaky_f1, honest_f1) * 1.18], title='F1 score'),
)
fig.show()


**What we did.** Dropped `duration` from `X` and re-trained. F1 fell from `0.60` → `0.46`. The drop is the leakage tax — the 0.87 was scored against test rows whose `duration` had already been observed; the 0.40 is the model's actual ability to predict *before the call*.

**Why drop, and not one of the other three options?**
- *Caveat-and-keep* fails in practice — the caveat gets stripped when the model is exported and re-used by a colleague who skipped this notebook.
- *Build a pre-call duration proxy* would require its own pipeline and would inherit any error in that pipeline. Not worth it for v1; possibly worth it for v2.
- *Reframe the prediction time* changes the business problem (the bank wants to decide *whether to make the call*, not how to react mid-call).

Drop is the simplest correct move and the one the dataset documentation literally recommends.

**When you would NOT drop.** If you genuinely have a *post-event prediction* problem — e.g., predicting whether a call needs a human follow-up *after* the agent has logged it — then `duration` is legitimate. The diagnostic question is always the same: *could this feature exist before the prediction time?* For our problem the answer is no, so it goes.


### 1.4 — Sentinel handling: `pdays = 999`

`pdays` records *days since last contact*. But ~96% of rows have `pdays = 999`, which the documentation says means *"client was not previously contacted"*. **That 999 is a sentinel, not a real number.**


**Why this matters.** A sentinel value is a special code that means *"this measurement does not apply"* — here, `pdays = 999` means *"there was no previous contact"*, not *"the previous contact was 999 days ago"*. Treating it as a real number poisons everything downstream: `StandardScaler` computes a mean and std dominated by the 999s, the genuine numeric variation in the 4% who *were* previously contacted is squashed flat, and the model's view of the column collapses to *"is this row in the 96% bucket or the 4% bucket?"* — exactly equivalent to a binary flag, with all the temporal information thrown away.

**Our options:**

| Option | What it does | Trade-off |
|---|---|---|
| **Leave as-is (do nothing)** | `StandardScaler` runs on the raw column | Mean and std are corrupted by the sentinel; temporal information lost; *the silent-failure default* |
| **Drop the column** | Remove `pdays` entirely | Loses both the *contacted-before-or-not* signal AND the *recency* signal — too aggressive |
| **Mean / median impute the 999s** | Replace 999s with the median of the real days | Fabricates contact history for 96% of customers; introduces a fake "average" recency signal that does not exist |
| **Split into flag + clean numeric** ✓ | Add `was_contacted_before` (boolean) and `pdays_clean` (real days, NaN for never-contacted, then median-impute the NaNs) | Two columns instead of one; preserves *both* signals cleanly; the model can learn which one matters |

We will use **split into flag + clean numeric** below. The reasoning is in the cell after the code.


In [21]:
pdays_999 = (X_honest["pdays"] == 999).sum()
pdays_real = len(X_honest) - pdays_999
print(f"pdays=999 (never contacted) : {pdays_999:,}  ({pdays_999/len(X_honest)*100:.1f}%)")
print(f"pdays != 999 (real days)    : {pdays_real:,}")
print(f"\nIf we leave pdays as-is and run StandardScaler, the mean will be ~{X_honest['pdays'].mean():.1f}")
print(f"and the std will be ~{X_honest['pdays'].std():.1f} — both dominated by the sentinel.")
box("warning", "Sentinels silently corrupt scaling",
    "<code>StandardScaler</code> would compute mean and std over a column that is 96% the value 999. "
    "The 'scaled' feature becomes near-zero for never-contacted, near-1 for actually-contacted, "
    "and indistinguishable from a binary flag — all the temporal information is lost.")


pdays=999 (never contacted) : 39,673  (96.3%)
pdays != 999 (real days)    : 1,515

If we leave pdays as-is and run StandardScaler, the mean will be ~962.5
and the std will be ~186.9 — both dominated by the sentinel.


In [22]:
# Fix: split the information into a boolean flag + a clean numeric
X_honest = X_honest.copy()
X_honest["was_contacted_before"] = (X_honest["pdays"] != 999).astype(int)
X_honest["pdays_clean"] = X_honest["pdays"].where(X_honest["pdays"] != 999, np.nan)
X_honest["pdays_clean"] = X_honest["pdays_clean"].fillna(X_honest["pdays_clean"].median())
X_honest = X_honest.drop(columns=["pdays"])
print("After the fix:")
print(X_honest[["was_contacted_before", "pdays_clean"]].describe())


After the fix:
       was_contacted_before   pdays_clean
count          41188.000000  41188.000000
mean               0.036783      6.000534
std                0.188230      0.733342
min                0.000000      0.000000
25%                0.000000      6.000000
50%                0.000000      6.000000
75%                0.000000      6.000000
max                1.000000     27.000000


**What we did.** Replaced the single `pdays` column with two columns:
- `was_contacted_before` — boolean flag (1 if `pdays != 999`, else 0)
- `pdays_clean` — real days, with `NaN` for never-contacted rows, then median-imputed so `StandardScaler` works downstream

Then dropped the original `pdays`.

**Why split, and not one of the other three options?**
- *Leave as-is* corrupts the scaler and silently kills the recency signal.
- *Drop the column* throws away two real signals at once: was-contacted and how-recently. We need both.
- *Mean-impute* fabricates a contact history for 96% of customers. The model would see Rs.~600-day-ago contact for everyone — a confident lie.
- *Split* lets the model learn from each piece separately: a coefficient on `was_contacted_before` (how much does prior contact matter at all?) and a coefficient on `pdays_clean` (among contacted customers, how much does recency matter?).

**When you would NOT split.** If the sentinel rate is very low (say <2% of rows), splitting is overkill — a simple median-impute on the sentinel rows is fine and the column-count bloat isn't worth it. The 96% sentinel rate here is what makes splitting essential.


### 1.5 — Cyclical encoding for `month`

**Why this matters.** `month` is a *periodic* feature — December is followed by January, not by some far-away thirteenth month. A naive integer encoding (`Jan=1, Feb=2, ..., Dec=12`) tells a linear model that December (12) is **eleven steps away from January (1)**, when in reality they are **one step apart on the calendar**. The model loses the wrap-around. A campaign in December and a campaign in January will be treated as opposites instead of neighbours.

**Our options:**

| Option | What it does | Trade-off |
|---|---|---|
| **Integer / ordinal encoding** | `Jan=1, Dec=12` as a single numeric column | Simplest; but breaks the wrap-around — the model thinks Dec and Jan are far apart |
| **One-hot encoding** | 12 boolean columns, one per month | Preserves no ordering at all; the model has to re-discover that Dec and Jan are similar from data alone; column-count grows |
| **Cyclical sin/cos encoding** ✓ | Two columns: `sin(2π·month/12)`, `cos(2π·month/12)` — places each month on the unit circle | December and January end up numerically adjacent (the wrap-around works); 2 columns instead of 12; standard trick for any periodic feature |
| **Target / mean encoding by month** | Replace month with the mean of `y` for that month | Strong signal but classic target leakage if not done with leave-one-out; deferred to Phase 2 |

We will use **sin/cos cyclical encoding** below. The reasoning is in the cell after the code.


In [23]:
month_to_num = {"jan":1,"feb":2,"mar":3,"apr":4,"may":5,"jun":6,
                "jul":7,"aug":8,"sep":9,"oct":10,"nov":11,"dec":12}
m = X_honest["month"].map(month_to_num)
X_honest["month_sin"] = np.sin(2 * np.pi * m / 12)
X_honest["month_cos"] = np.cos(2 * np.pi * m / 12)

fig = px.scatter(
    pd.DataFrame({"month_num": m, "sin": X_honest["month_sin"], "cos": X_honest["month_cos"]}).drop_duplicates(),
    x="sin", y="cos", text="month_num",
    title="Cyclical encoding places December next to January on the unit circle",
)
fig.update_traces(textposition="top right", marker=dict(size=14, color=MASAI_RED))
fig.update_layout(template="plotly_white", height=400, width=420,
                  xaxis=dict(range=[-1.3, 1.3], title="sin(2π·month/12)"),
                  yaxis=dict(range=[-1.3, 1.3], title="cos(2π·month/12)", scaleanchor="x", scaleratio=1))
fig.show()
box("definition", "Cyclical encoding for periodic features",
    "A naive integer encoding (Jan=1, Dec=12) tells the model December is 'far from' January, which is wrong — they are adjacent on a circle. "
    "<code>(sin, cos)</code> of <code>2π × month / 12</code> makes December and January numerically adjacent. "
    "Same trick for <code>day_of_week</code>, <code>hour_of_day</code>, <code>day_of_year</code>.")


**What we did.** Mapped each month name to its number (1–12), then computed two columns: `month_sin = sin(2π × month / 12)` and `month_cos = cos(2π × month / 12)`. Each month now sits on the unit circle, with December next to January as it should be.

**Why sin/cos, and not one of the other three options?**
- *Integer encoding* — breaks the wrap-around. A linear model sees `Dec=12, Jan=1` and learns that the December-to-January transition is an 11-unit jump in the wrong direction.
- *One-hot encoding* — would work (the model would re-learn similarity from data), but produces 12 columns instead of 2, and the model gets no help from us about the underlying geometry.
- *Target encoding* — strong signal but an easy way to leak the target if not done carefully. We avoid it in v1.

Sin/cos is the smallest, most-informative encoding for any periodic feature.

**When you would NOT use sin/cos.** If the period is irregular or unknown (e.g., a custom 13-month accounting calendar), sin/cos with the wrong period is worse than one-hot. The same trick generalises to *any* known period — `day_of_week` with period 7, `hour_of_day` with period 24, `day_of_year` with period 365 — but you need to know the period.


### 1.6 — Frequency encoding (when OneHotEncoder would explode)

**Why this matters.** `job` has 12 categories — `OneHotEncoder` handles that fine. But many real datasets have *high-cardinality* categoricals: thousands of cities, hundreds of products, tens of thousands of merchants. One-hot encoding a 5,000-category column produces 5,000 sparse columns, blows up memory, and gives the model no help in distinguishing common categories from rare ones. Frequency encoding is the standard fix — and it's worth practicing on `job` (where one-hot would still work) so the muscle is there when you face `merchant_id` (where it won't).

**Our options for a high-cardinality categorical:**

| Option | What it does | Trade-off |
|---|---|---|
| **OneHotEncoder** | One boolean column per category | Standard for low-cardinality; explodes column count for high-cardinality; gives the model zero prior about which categories matter |
| **Frequency encoding** ✓ | Replace each category with its row-count in train | One column instead of n; common-vs-rare is now numerically encoded; downside: the column has a fake ordinal interpretation (the model sees counts ordered) |
| **Target / mean encoding** | Replace each category with the mean of `y` for that category | Strongest signal possible; *direct target leakage* unless done with cross-fold averaging — risky. Deferred to Phase 2 |
| **Hashing trick (FeatureHasher)** | Hash category names into a fixed number of buckets | Bounded column count regardless of cardinality; collisions inevitable; debug-hostile |
| **Embeddings (deep learning)** | Learn a vector per category as part of training | Best for very high cardinality + lots of data; out of scope for sklearn linear models |

We will use **frequency encoding** on `job` below as a demonstration. The reasoning is in the cell after the code.


In [24]:
# job has 12 categories — fine for OneHot. But what if it had 1000?
job_counts = X_honest["job"].value_counts()
print(f"`job` has {len(job_counts)} categories — OneHotEncoder would produce {len(job_counts)} columns:")
print(job_counts.head(8))
X_honest["job_freq"] = X_honest["job"].map(job_counts)
print("\nAfter frequency encoding (one column instead of 12):")
print(X_honest[["job", "job_freq"]].head())
box("definition", "Frequency encoding — when to reach for it",
    "For high-cardinality categoricals (1000s of cities, 100s of products), <code>OneHotEncoder</code> blows up the column count. "
    "<b>Frequency encoding</b> replaces each category with its count. One column instead of n. "
    "Trade-off: the resulting numeric has a fake ordinal interpretation (counts are ordered), so be wary on linear models.")


`job` has 12 categories — OneHotEncoder would produce 12 columns:
job
admin.           10422
blue-collar       9254
technician        6743
services          3969
management        2924
retired           1720
entrepreneur      1456
self-employed     1421
Name: count, dtype: int64

After frequency encoding (one column instead of 12):
         job  job_freq
0  housemaid      1060
1   services      3969
2   services      3969
3     admin.     10422
4   services      3969


**What we did.** Computed `value_counts()` on `job` and added `job_freq` — each row's value is the count of how many listings share that customer's job category. One column instead of twelve.

**Why frequency, and not one of the other four options?**
- *OneHotEncoder* would still work for `job` (12 categories is fine). We're doing frequency here as a *demonstration of the pattern* you would reach for when the column has 1,000+ categories.
- *Target encoding* gives the strongest signal but is direct target leakage unless done with leave-one-out cross-folding. Risky for v1; deferred to Phase 2.
- *Feature hashing* makes sense at very high cardinality (10k+) where even `value_counts()` is expensive. Overkill here.
- *Embeddings* require deep-learning machinery; out of scope for a linear-model session.

Frequency encoding is the workhorse middle ground: cheap, leakage-safe, gives the model a meaningful signal (common vs rare) in one column.

**When you would NOT use frequency encoding.** If the categorical is *truly nominal with no ordering implied by frequency* — e.g., `country_code` where Brazil being more common than Norway shouldn't push Brazil 'higher' than Norway numerically — frequency encoding can confuse a linear model. In that case, prefer one-hot (low cardinality) or target encoding with proper cross-folding (high cardinality).


### 1.7b — Re-split with engineered features

We added new columns to `X_honest` after the Section 1.3 split — so `X_tr_h` and `X_te_h` are now stale (they don't have the new columns). Re-split now so the rest of the lab uses the enriched DataFrame everywhere.


In [25]:
X_tr_h, X_te_h, y_tr_h, y_te_h = train_test_split(
    X_honest, y, stratify=y, test_size=0.2, random_state=RNG,
)
print(f"X_honest shape: {X_honest.shape}")
print(f"X_tr_h shape  : {X_tr_h.shape}  (now includes all engineered columns)")
print(f"New columns added since Section 1.3:")
for c in ["was_contacted_before", "pdays_clean", "month_sin", "month_cos", "job_freq"]:
    print(f"  {c}: {'present' if c in X_tr_h.columns else 'MISSING'}")


X_honest shape: (41188, 23)
X_tr_h shape  : (32950, 23)  (now includes all engineered columns)
New columns added since Section 1.3:
  was_contacted_before: present
  pdays_clean: present
  month_sin: present
  month_cos: present
  job_freq: present


### 1.7 — The 6-pattern feature creation library (cheat sheet)

| Pattern | When | Example |
|---|---|---|
| **Extract** | Date or composite columns | `df['date'].dt.year`, `df['date'].dt.day_of_week`, cyclical `(sin, cos)` |
| **Transform** | Skewed numeric | `np.log1p(amount)`, `np.sqrt(count)`, Box-Cox |
| **Combine** | Domain-relevant ratios | `price_per_sqft = price / sqft`, `engagement_rate = clicks / impressions` |
| **Bin** | Continuous → categorical | `pd.cut(age, [0,25,40,60,100])`, `pd.qcut(income, q=5)` |
| **Sentinel** | Special-value codes | `pdays=999` → `was_contacted` boolean + clean numeric |
| **Frequency-encode** | High-cardinality categorical | `df['city_freq'] = df['city'].map(df['city'].value_counts())` |


---

## Section 2 — `ColumnTransformer` Deep Dive

The only correct way to preprocess mixed types.

### 2.1 — The pre-`ColumnTransformer` manual pain (do NOT do this)

Three subtle bugs in five lines. None will raise an error. All will silently produce wrong predictions in production.


In [28]:
# WRONG — manual concatenation, leakage-prone, brittle.
# This block is for illustration; we DO NOT use this pattern downstream.
#
# X_num_tr = StandardScaler().fit_transform(X_tr_h[num_cols_h])    # OK on train
# X_cat_tr = pd.get_dummies(X_tr_h[cat_cols_h])                     # OK on train
# X_train_proc = np.hstack([X_num_tr, X_cat_tr.values])             # Train ready
#
# # At test time you have to repeat ALL of this manually — and three bugs creep in:
# X_num_te = StandardScaler().fit_transform(X_te_h[num_cols_h])     # ⚠ BUG: re-fits on TEST → leakage
# X_cat_te = pd.get_dummies(X_te_h[cat_cols_h])                     # ⚠ BUG: missing/extra columns
# X_test_proc = np.hstack([X_num_te, X_cat_te.values])              # ⚠ BUG: shape mismatch silent

box("danger", "Why we need ColumnTransformer",
    "The manual approach has three production-killing bugs:<br>"
    "1. <b>Re-fitting StandardScaler on test data</b> uses test statistics — leakage.<br>"
    "2. <b><code>pd.get_dummies</code> on the test set</b> can produce different columns than train (e.g., a category not present in test).<br>"
    "3. <b>Shape mismatches</b> are silent until they aren't. "
    "<code>ColumnTransformer + Pipeline</code> make all three impossible by construction.")


#### How `ColumnTransformer` routes columns — visual

The Sankey diagram below shows what `ColumnTransformer` does conceptually: numeric columns flow into the scaler branch, categorical columns flow into the OneHotEncoder branch, and the outputs concatenate into the feature matrix the estimator sees. Hover any link for column counts.


In [29]:
src_labels = (['Numeric input'] + ['Categorical input'])
branch_labels = ['StandardScaler', 'OneHotEncoder']
out_label = ['Output feature matrix']

labels = src_labels + branch_labels + out_label
# indices: 0=Numeric, 1=Categorical, 2=Scaler, 3=OHE, 4=Output
n_num = len(num_cols)
n_cat = len(cat_cols)
ohe_out_estimate = sum(df[c].nunique() for c in cat_cols)

sankey = go.Figure(data=[go.Sankey(
    arrangement='snap',
    node=dict(
        pad=24, thickness=24, line=dict(color='black', width=0.6),
        label=labels,
        color=['#1565C0', '#2E7D32', '#1565C0', '#2E7D32', MASAI_RED],
    ),
    link=dict(
        source=[0, 1, 2, 3],
        target=[2, 3, 4, 4],
        value=[n_num, n_cat, n_num, ohe_out_estimate],
        color=['rgba(21,101,192,0.35)', 'rgba(46,125,50,0.35)', 'rgba(21,101,192,0.55)', 'rgba(46,125,50,0.55)'],
        hovertemplate='%{source.label} \u2192 %{target.label}<br>cols=%{value}<extra></extra>',
    ))])
sankey.update_layout(
    title=f'ColumnTransformer flow: {n_num} numeric \u2192 scaler, {n_cat} categorical \u2192 OHE (\u2248 {ohe_out_estimate} OHE columns)',
    template='plotly_white', height=380, font=dict(size=12),
)
sankey.show()


### 2.2 — The corrected pattern: `ColumnTransformer` constructor

**Why this matters.** Section 2.1 just showed the manual pre-`ColumnTransformer` pain — `pd.get_dummies` + `StandardScaler.fit_transform` + `pd.concat` + manual column alignment between train and test. That code is *easy to get wrong* and *hard to debug*: a category present in train but missing in test silently produces a different column count, the scaler fitted on train must be carefully re-applied (not re-fitted) to test, and any change to the column list cascades through three places. We need a single object that owns the routing — *numeric columns to the scaler branch, categorical columns to the one-hot branch, output concatenated automatically*.

**Our options for routing different transforms to different columns:**

| Option | What it does | Trade-off |
|---|---|---|
| **Manual `pd.concat` + separate fit_transform calls** | What 2.1 just did | Three places to keep aligned; train/test asymmetry is silent; column-list changes cascade |
| **`FunctionTransformer` per column** | Wrap each per-column transform in a `FunctionTransformer` and chain | Verbose; loses sklearn's introspection (no `get_feature_names_out`) |
| **`ColumnTransformer`** ✓ | One object holding `(name, transformer, columns)` tuples; routes columns automatically | The sklearn-blessed answer (added v0.20, Dec 2018); composable with `Pipeline`; supports `set_output('pandas')`; gives you `get_feature_names_out` |
| **`make_column_transformer`** (the helper) | Same as `ColumnTransformer` but auto-names the steps | Less explicit (auto-names like `standardscaler-1`); we use the constructor for explicit step names |

We will use the **`ColumnTransformer` constructor** below. The reasoning is in the cell after the code.


In [32]:
# Re-derive numeric/categorical lists for the post-Section 1 X_honest dataframe
X_honest = X_honest.drop(columns=["month", "job"])  # drop the original month column after cyclical encoding
num_cols_final = X_honest.select_dtypes(include="number").columns.tolist()
cat_cols_final = X_honest.select_dtypes(include="object").columns.tolist()
print(f"Final numeric ({len(num_cols_final)}):    {num_cols_final}")
print(f"Final categorical ({len(cat_cols_final)}): {cat_cols_final}")


Final numeric (13):    ['age', 'campaign', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'was_contacted_before', 'pdays_clean', 'month_sin', 'month_cos', 'job_freq']
Final categorical (8): ['marital', 'education', 'default', 'housing', 'loan', 'contact', 'day_of_week', 'poutcome']


In [33]:
preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(),                                 num_cols_final),
        ("cat", OneHotEncoder(handle_unknown="ignore", drop=None, sparse_output=False), cat_cols_final),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)
print("ColumnTransformer constructed:")
print(preprocess)
box("definition", "Tuple anatomy: (name, transformer, columns)",
    "Each tuple in <code>transformers=[...]</code> is <code>(name, transformer, columns)</code>. "
    "Names must be unique. <code>remainder='drop'</code> discards columns not listed (default; safe). "
    "<code>verbose_feature_names_out=False</code> strips the <code>num__</code>/<code>cat__</code> prefixes from output column names.")


ColumnTransformer constructed:
ColumnTransformer(transformers=[('num', StandardScaler(),
                                 ['age', 'campaign', 'previous', 'emp.var.rate',
                                  'cons.price.idx', 'cons.conf.idx',
                                  'euribor3m', 'nr.employed',
                                  'was_contacted_before', 'pdays_clean',
                                  'month_sin', 'month_cos', 'job_freq']),
                                ('cat',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 ['marital', 'education', 'default', 'housing',
                                  'loan', 'contact', 'day_of_week',
                                  'poutcome'])],
                  verbose_feature_names_out=False)


**What we did.** Built a single `ColumnTransformer` with two named branches: `"num"` routes the numeric columns to `StandardScaler`, `"cat"` routes the categorical columns to `OneHotEncoder(handle_unknown='ignore', sparse_output=False)`. `remainder='drop'` discards anything unlisted. `verbose_feature_names_out=False` keeps output column names clean.

**Why the constructor, and not one of the other three options?**
- *Manual concat* (2.1) is what we are explicitly replacing — silent failure mode, train/test asymmetry, fragile.
- *FunctionTransformer chains* lose `get_feature_names_out` and don't compose with `Pipeline` cleanly.
- *`make_column_transformer`* works but auto-generates step names like `standardscaler-1` — fine for quick prototypes but harder to read in production logs and harder to address with `Pipeline.set_params("<step>__<param>", ...)` syntax (which we use in 4.3).

The explicit constructor wins on readability + addressability + composability.

**When you would NOT use `ColumnTransformer`.** If your dataset is purely numeric (the 12.1 credit-card-fraud case), a single `StandardScaler` is enough — `ColumnTransformer` would be over-engineered for one branch. Mixed-type datasets are where it earns its keep.


### 2.3 — `remainder='drop'` vs `'passthrough'`

| Setting | Behavior | Use when |
|---|---|---|
| `remainder='drop'` (default) | Columns not listed are silently discarded | You want the transformer's column lists to be authoritative |
| `remainder='passthrough'` | Columns not listed are concatenated unchanged into the output | You have a numeric column already on a uniform scale that should bypass scaling |


### 2.4 — `make_column_selector` for dynamic typing

In [34]:
preprocess_dyn = ColumnTransformer([
    ("num", StandardScaler(),                            make_column_selector(dtype_include=np.number)),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), make_column_selector(dtype_include="object")),
])
preprocess_dyn.fit(X_honest)
print("`make_column_selector(dtype_include=np.number)` selects:",
      list(X_honest.select_dtypes(include=np.number).columns))
box("tip", "Dynamic vs explicit",
    "<code>make_column_selector</code> auto-adapts when the schema changes (new columns added, dtypes shift). "
    "Explicit lists are easier to grep and audit. Pick whichever matches your team's discipline; both are valid.")


`make_column_selector(dtype_include=np.number)` selects: ['age', 'campaign', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'was_contacted_before', 'pdays_clean', 'month_sin', 'month_cos', 'job_freq']


### 2.5 — `set_output(transform='pandas')` (sklearn 1.2+)

In [35]:
preprocess.set_output(transform="pandas")
X_processed = preprocess.fit_transform(X_tr_h)
print(f"Output type: {type(X_processed).__name__}")
print(f"Output shape: {X_processed.shape}")
print(f"\nFirst 5 column names:\n{list(X_processed.columns[:5])}")
print(f"\nLast 5 column names:\n{list(X_processed.columns[-5:])}")
box("tip", "Why this matters",
    "Without <code>set_output(transform='pandas')</code>, the output is a plain numpy array and column names are lost. "
    "With it, you keep DataFrame columns flowing through the Pipeline — invaluable for debugging and inspection.")


Output type: DataFrame
Output shape: (32950, 44)

First 5 column names:
['age', 'campaign', 'previous', 'emp.var.rate', 'cons.price.idx']

Last 5 column names:
['day_of_week_tue', 'day_of_week_wed', 'poutcome_failure', 'poutcome_nonexistent', 'poutcome_success']


### 2.6 — `get_feature_names_out()` for inspection after fit

In [37]:
feature_names = preprocess.get_feature_names_out()
print(f"Total output features: {len(feature_names)}")
print(f"\nFirst 10:\n{feature_names[:10]}")
print(f"\nLast 10:\n{feature_names[-10:]}")
box("definition", "Inspecting your fitted Pipeline's actual columns",
    "After <code>pipe.fit(X_train, y_train)</code>, you can ask the preprocessor what features it produced. "
    "Pair with <code>pipe.named_steps['clf'].coef_</code> to inspect coefficients on the actual transformed columns. "
    "This is the single most useful debugging tool when something goes wrong.")


Total output features: 44

First 10:
['age' 'campaign' 'previous' 'emp.var.rate' 'cons.price.idx'
 'cons.conf.idx' 'euribor3m' 'nr.employed' 'was_contacted_before'
 'pdays_clean']

Last 10:
['contact_cellular' 'contact_telephone' 'day_of_week_fri'
 'day_of_week_mon' 'day_of_week_thu' 'day_of_week_tue' 'day_of_week_wed'
 'poutcome_failure' 'poutcome_nonexistent' 'poutcome_success']


In [38]:
box("tip", "The four sklearn 1.2+ knobs you must know",
    "1. <code>OneHotEncoder(sparse_output=False)</code> — replaces deprecated <code>sparse=False</code>.<br>"
    "2. <code>OneHotEncoder(handle_unknown='ignore')</code> — production safeguard for unseen categories.<br>"
    "3. <code>set_output(transform='pandas')</code> — keeps DataFrame columns flowing through the Pipeline.<br>"
    "4. <code>get_feature_names_out()</code> — debugging your fitted Pipeline's actual columns.")


---

## Section 3 — Interaction Features

Letting linear models see what trees see.

### 3.1 — Why interactions matter for linear models

Linear models can capture `y = β₀ + β₁·age + β₂·balance` but cannot natively capture `y = β·(age × balance)`. Tree-based models can — that's why trees often outperform untuned linear models. **Adding interaction features lets a linear model see what a tree sees.**


**Why this matters.** A bare linear model is *additive*: it can say *"older customers subscribe more"* and *"customers with high balances subscribe more"*, but it cannot say *"older customers WITH high balances subscribe especially more"* — that's an interaction effect, and it requires the product term `age × balance` to be in the feature matrix. Trees discover interactions natively (each split conditional on a previous split is an interaction). For a linear model, you have to engineer the interaction in.

**Our options for adding interactions:**

| Option | What it does | Trade-off |
|---|---|---|
| **Switch to a tree-based model** | RandomForest / GradientBoosting / XGBoost capture interactions natively | Loses linear-model interpretability + the `coef_` table; deferred to Module 2 |
| **Hand-craft domain interactions** | E.g., `debt_to_income = debt / income`; one ratio per business hypothesis | Interpretable, low-dimensional, no multicollinearity; but you only see what you think to look for |
| **`PolynomialFeatures(degree=2, interaction_only=True)`** ✓ | Auto-generate every pairwise product among the `n` input features | Catches interactions you didn't think of; output count grows as n + n(n-1)/2 — manageable for small n, explosive for large n |
| **`PolynomialFeatures(degree=3+)` or `interaction_only=False`** | Higher-order terms (a²·b, a·b·c) | Combinatorial explosion; usually overkill unless paired with strong regularization |

We will use **`PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)`** below — the sweet spot for letting a linear model see all pairwise interactions without going to cubic-and-above terms. The reasoning is in the cell after the code.


In [43]:
from sklearn.preprocessing import PolynomialFeatures
poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
out = poly.fit_transform([[2, 3, 5]])
print("Input: [2, 3, 5]  (3 features: a, b, c)")
print(f"Output: {out[0].tolist()}  (6 features: a, b, c, ab, ac, bc)")
print("Output names:", poly.get_feature_names_out(["a", "b", "c"]))


Input: [2, 3, 5]  (3 features: a, b, c)
Output: [2.0, 3.0, 5.0, 6.0, 10.0, 15.0]  (6 features: a, b, c, ab, ac, bc)
Output names: ['a' 'b' 'c' 'a b' 'a c' 'b c']


In [44]:
# Combinatorial blow-up
ns = [3, 5, 10, 20, 50, 100]
rows = []
for n in ns:
    pairs = n * (n - 1) // 2
    rows.append({"n input features": n, "pairs n(n-1)/2": pairs, "total output": n + pairs})
blowup = pd.DataFrame(rows)
blowup


,n input features,pairs n(n-1)/2,total output
0,3,3,6
1,5,10,15
2,10,45,55
3,20,190,210
4,50,1225,1275
5,100,4950,5050


In [47]:
# Interactive polynomial degree explorer (Tier 2 / 12.2 analog)
n_choices = [2, 3, 5, 10, 15, 20, 30, 50, 75, 100]
all_n = np.arange(2, 101)
all_total = all_n + all_n * (all_n - 1) // 2

fig = go.Figure()
fig.add_trace(go.Scatter(x=all_n, y=all_total, mode='lines',
    line=dict(color=MASAI_RED, width=3), name='n + n(n-1)/2',
    hovertemplate='n=%{x}<br>total output=%{y}<extra></extra>'))
init_n = n_choices[3]
init_total = init_n + init_n*(init_n-1)//2
fig.add_trace(go.Scatter(x=[init_n], y=[init_total], mode='markers+text',
    marker=dict(color='#1565C0', size=16, symbol='diamond'),
    text=[f' n={init_n} \u2192 {init_total} features'], textposition='top right',
    name='Selected n',
    hovertemplate=f'n={init_n}<br>total={init_total}<extra></extra>'))

frames = []
for n in n_choices:
    pairs = n * (n - 1) // 2
    total = n + pairs
    frames.append(go.Frame(name=f'{n}', data=[
        go.Scatter(x=all_n, y=all_total, mode='lines',
                   line=dict(color=MASAI_RED, width=3)),
        go.Scatter(x=[n], y=[total], mode='markers+text',
                   marker=dict(color='#1565C0', size=16, symbol='diamond'),
                   text=[f' n={n} \u2192 {total} features'], textposition='top right'),
    ], layout=go.Layout(annotations=[dict(
        text=f"<b>n input features</b>: {n}<br><b>+ pairs n(n-1)/2</b>:  {pairs:,}<br><b>= total output</b>:  {total:,}",
        xref='paper', yref='paper', x=0.02, y=0.98, xanchor='left', yanchor='top',
        showarrow=False, align='left', font=dict(family='monospace', size=12),
        bgcolor='#e3f2fd', bordercolor='#1565C0', borderwidth=1, borderpad=8)])))

fig.frames = frames
fig.update_layout(
    title='PolynomialFeatures(degree=2, interaction_only=True) \u2014 drag n to see the blow-up',
    template='plotly_white', height=420, showlegend=False,
    margin=dict(l=60, r=40, t=80, b=120),
    transition=dict(duration=400, easing='cubic-in-out'),
    annotations=[dict(
        text=f"<b>n input features</b>: {init_n}<br><b>+ pairs n(n-1)/2</b>:  {init_n*(init_n-1)//2:,}<br><b>= total output</b>:  {init_total:,}",
        xref='paper', yref='paper', x=0.02, y=0.98, xanchor='left', yanchor='top',
        showarrow=False, align='left', font=dict(family='monospace', size=12),
        bgcolor='#e3f2fd', bordercolor='#1565C0', borderwidth=1, borderpad=8)],
    xaxis_title='n input features', yaxis_title='Total output features',
    yaxis=dict(type='log'),
    sliders=[dict(active=3, x=0.05, y=-0.15, len=0.9, pad=dict(t=20, b=10),
        currentvalue=dict(prefix='n = ', font=dict(size=14, color='#1565C0')),
        steps=[dict(method='animate', label=f'{n}',
            args=[[f'{n}'], dict(mode='immediate',
                                  frame=dict(duration=400, redraw=True),
                                  transition=dict(duration=400, easing='cubic-in-out'))])
               for n in n_choices])],
)
fig.show()
box('math', 'Output count formula',
    'Output features = $n + \\binom{n}{2} = n + \\dfrac{n(n-1)}{2}$.<br>'
    'For n=10: 55. For n=20: 210. For n=50: 1,275. For n=100: 5,050.<br>'
    'Combine with OneHotEncoder expanding 10 categoricals \u00d7 8 levels each = 80 columns, and n becomes 90 \u2192 output \u2248 4,095 features. '
    "<b>Don't blindly polynomial-expand a wide dataset.</b>")


**What we did.** Demonstrated `PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)` and visualised the combinatorial blow-up for `n` ranging from 2 to 100. With `n=10` input features the output is 55 columns; with `n=20` it's 210; with `n=100` it's 5,050.

**Why `degree=2, interaction_only=True`, and not the other three options?**
- *Switch to trees* would catch interactions natively, but loses the linear model's `coef_`-table interpretability — the whole reason we picked Logistic Regression. Trees are a Module 2 conversation.
- *Hand-crafted domain interactions* is excellent when you have a clear hypothesis (and we cover this in 3.2) — but you only see what you think to look for. PolynomialFeatures catches interactions you didn't anticipate.
- *`degree=3+` or `interaction_only=False`* (which adds `a²`, `b²`, ...) explodes the column count and rarely helps without strong regularization. `degree=2, interaction_only=True` is the well-known sweet spot.

**When you would NOT use PolynomialFeatures.** If `n` is already large (50+ numeric features), the output is 1,275+ columns — too many for an unregularized model, and slow to fit. Pair it with `Lasso` regularization (Session 13.2) so the model can prune the useless interactions. If you have <20 features and a clear domain hypothesis, hand-crafted ratios (Section 3.2) usually beat automatic expansion on interpretability + stability.


### 3.2 — Manual domain interactions vs automatic

Both are valid. Heuristic:

- **Manual** when domain suggests it (`debt_to_income = debt / income`). One feature, interpretable, no multicollinearity.
- **`PolynomialFeatures`** when you don't know what to look for and have regularization downstream (Lasso in 13.2 will prune).


In [48]:
box("definition", "Forward reference: multicollinearity",
    "Adding <code>a·b</code> and <code>a·c</code> as features when <code>b</code> and <code>c</code> are correlated produces highly correlated polynomial features. "
    "This inflates standard errors and destabilises coefficient estimates — the canonical multicollinearity problem. "
    "<b>14.1 (Advanced Regression)</b> introduces VIF for diagnosis; <b>13.2 (Regularization)</b> introduces Ridge/Lasso for the cure.")


## Section 4 — The Full Sklearn Pipeline

ColumnTransformer + estimator → one fit, one predict, no leakage.

### 4.1 — Compose `ColumnTransformer + LogisticRegression`

**Why this matters.** We have `ColumnTransformer` (preprocessing) and `LogisticRegression` (the estimator) as two separate objects. We need to *compose* them into a single object that takes raw `X` in and predictions out — for three reasons:
1. **Leakage prevention.** When sklearn's `cross_val_score` or `GridSearchCV` calls `pipe.fit()` on a fold, only that fold's training portion sees `preprocess.fit_transform()`. The fold's test portion sees only `preprocess.transform()`. *No statistic is ever fit on test data.* This is mechanically impossible to leak.
2. **Deployment as one artifact.** `joblib.dump(pipe, ...)` saves the encoder's category lists, the scaler's mean/std, and the model's coefficients into a single file. The 20.2 FastAPI inference endpoint loads this one file.
3. **Hyperparameter tuning across the whole pipeline.** `GridSearchCV` can search over `clf__C` *and* `preprocess__num__with_mean` in one call (covered in 16.2).

**Our options for combining preprocessing + estimator:**

| Option | What it does | Trade-off |
|---|---|---|
| **Manual chain (.fit_transform → .fit)** | Call each step yourself in train order, mirror it in test order | Three places to keep aligned (train, test, deployment); leakage-easy in CV; no single artefact to save |
| **`sklearn.pipeline.Pipeline`** ✓ | One object holding `[(name, step), ...]`; calls `fit` / `transform` / `predict` automatically in order | Standard sklearn answer; leakage-safe in CV; one `joblib.dump` saves everything |
| **`imblearn.pipeline.Pipeline`** | Like sklearn's, but supports SMOTE-style resamplers as a step | Use when you have an imbalanced-resampler step (we did in 12.1). Bank Marketing is moderate imbalance; we use `class_weight='balanced'` instead, so the sklearn version is enough. |
| **`make_pipeline`** (helper) | Same as `Pipeline` but auto-names steps (`logisticregression`, `columntransformer`, ...) | Less explicit step names; harder to address with `<step>__<param>` syntax |

We will use **`sklearn.pipeline.Pipeline`** with explicit step names (`"preprocess"` and `"clf"`) below. The reasoning is in the cell after the code.


In [49]:
preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(),                                 num_cols_final),
        ("cat", OneHotEncoder(handle_unknown="ignore", drop=None, sparse_output=False), cat_cols_final),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)
preprocess.set_output(transform="pandas")

pipe = Pipeline([
    ("preprocess", preprocess),
    ("clf",        LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RNG)),
])
pipe.fit(X_tr_h, y_tr_h)
y_pred = pipe.predict(X_te_h)
y_proba = pipe.predict_proba(X_te_h)[:, 1]
print(f"F1 (subscribe class) : {f1_score(y_te_h, y_pred):.3f}")
print(f"PR-AUC               : {average_precision_score(y_te_h, y_proba):.3f}")
print(f"ROC-AUC              : {roc_auc_score(y_te_h, y_proba):.3f}")


F1 (subscribe class) : 0.418
PR-AUC               : 0.451
ROC-AUC              : 0.793


**What we did.** Wrapped the `ColumnTransformer` (now with `set_output('pandas')` for debugability) and a `LogisticRegression(class_weight='balanced')` into a single `Pipeline` with two named steps: `"preprocess"` and `"clf"`. Called `pipe.fit(X_tr_h, y_tr_h)` once. Called `pipe.predict(X_te_h)` once. No manual `fit_transform`, no concat, no train/test alignment code.

**Why `sklearn.pipeline.Pipeline` with explicit names, and not the other three options?**
- *Manual chain* is what we have been replacing throughout this session — silent leakage in CV, fragile, three-place updates, no single artefact. The thing we are trying to never do again.
- *`imblearn.pipeline.Pipeline`* is right when you have a resampling step like SMOTE (12.1). For moderate imbalance, `class_weight='balanced'` does the job and we don't need imblearn's machinery here.
- *`make_pipeline`* is fine for one-off prototypes but auto-names like `logisticregression` make the `<step>__<param>` syntax awkward and the diagnostic logs harder to read. Explicit naming pays off the moment you need to tune hyperparameters (16.2) or ship to production (20.2).

**When you would NOT use `sklearn.pipeline.Pipeline`.** If you need a custom training loop (deep learning), or your preprocessing needs *online* statistics that `Pipeline` can't express, you reach for PyTorch / TF or a custom estimator. For 99% of tabular sklearn work — including everything in this course — `Pipeline` is the right answer. The next 10 sessions assume you have it in your toolbox.


### 4.2 — Cross-validation: leakage-free by construction

In [50]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RNG)
scores = cross_val_score(pipe, X_honest, y, cv=cv, scoring="average_precision")
print(f"5-fold CV PR-AUC: {scores.mean():.3f} ± {scores.std():.3f}")
print(f"Per-fold scores : {[f'{s:.3f}' for s in scores]}")
box("definition", "Why this is leakage-free",
    "<code>cross_val_score</code> calls <code>pipe.fit(X_train_fold, y_train_fold)</code> on each fold, which calls "
    "<code>preprocess.fit(X_train_fold)</code> — the scaler's mean and the encoder's category list are learned ONLY from that fold's training portion. "
    "The fold's test portion sees <code>preprocess.transform(X_test_fold)</code>, applying statistics from the train fold only. "
    "<b>It is mechanically impossible to leak.</b>")


5-fold CV PR-AUC: 0.437 ± 0.021
Per-fold scores : ['0.415', '0.424', '0.474', '0.429', '0.441']


### 4.3 — Hyperparameter access via `<step>__<param>` syntax (preview of 16.2)

In [51]:
# Update parameters of any step using the namespaced syntax
pipe.set_params(clf__C=0.5, preprocess__num__with_mean=False)
print("Updated params: clf__C=0.5, preprocess__num__with_mean=False")
pipe.fit(X_tr_h, y_tr_h)
print(f"Refit F1: {f1_score(y_te_h, pipe.predict(X_te_h)):.3f}")
# Reset for downstream cells
pipe.set_params(clf__C=1.0, preprocess__num__with_mean=True)
box("tip", "Forward reference to 16.2 — Hyperparameter Tuning",
    "In 16.2 you'll write:<br>"
    "<code>GridSearchCV(pipe, {'clf__C': [0.01, 0.1, 1.0, 10.0], 'preprocess__num__with_mean': [True, False]})</code><br>"
    "That syntax requires today's Pipeline structure. <b>You won't learn GridSearchCV today; you will use this Pipeline today, which is the foundation.</b>")


Updated params: clf__C=0.5, preprocess__num__with_mean=False
Refit F1: 0.421


---

## Section 5 — Model Serialization (joblib vs pickle)

The production hand-off.

### 5.1 — `joblib.dump` and `joblib.load`

In [53]:
# Refit cleanly
pipe.fit(X_tr_h, y_tr_h)
joblib.dump(pipe, "model_uncompressed.joblib")
joblib.dump(pipe, "model.joblib", compress=3)

import os
uncompressed_mb = os.path.getsize("model_uncompressed.joblib") / 1e6
compressed_mb = os.path.getsize("model.joblib") / 1e6
print(f"Uncompressed: {uncompressed_mb:.3f} MB")
print(f"Compressed   : {compressed_mb:.3f} MB ({compressed_mb/uncompressed_mb*100:.0f}% of uncompressed)")


Uncompressed: 0.008 MB
Compressed   : 0.003 MB (35% of uncompressed)


### 5.2 — Round-trip verification: load in a fresh state, predict, compare

In [55]:
# Forget the in-memory pipeline; load from disk
pipe_loaded = joblib.load("model.joblib")
y_pred_loaded = pipe_loaded.predict(X_te_h)
y_pred_inmem  = pipe.predict(X_te_h)

match = np.array_equal(y_pred_loaded, y_pred_inmem)
print(f"Predictions match across fresh load: {match}")
f1_loaded = f1_score(y_te_h, y_pred_loaded)
print(f"F1 from loaded model: {f1_loaded:.3f}")
assert match, "Round-trip mismatch — serialization is broken"
box("output", "Round-trip verified",
    f"The loaded Pipeline produces identical predictions to the in-memory Pipeline. "
    f"F1 from loaded model = <b>{f1_loaded:.3f}</b>. <b>This is what gets shipped to production.</b>")


Predictions match across fresh load: True
F1 from loaded model: 0.418


### 5.3 — `pickle` vs `joblib`

In [56]:
import pickle
import time

t0 = time.time(); _ = pickle.dumps(pipe); t_pickle = time.time() - t0
t0 = time.time(); _ = joblib.dump(pipe, "_tmp.joblib"); t_joblib = time.time() - t0

print(f"pickle.dumps  : {t_pickle*1000:.1f} ms")
print(f"joblib.dump   : {t_joblib*1000:.1f} ms")
os.remove("_tmp.joblib")
box("definition", "When to use which",
    "<b>joblib</b> — preferred for sklearn models. Optimised for numpy arrays; supports built-in compression with <code>compress=3</code>.<br>"
    "<b>pickle</b> — Python stdlib; fully general; slower on large numpy. Fine for small Python objects without numpy.")


pickle.dumps  : 0.3 ms
joblib.dump   : 2.4 ms


### 5.4 — The security warning

In [57]:
box("danger", "Never load a joblib/pickle file from an untrusted source",
    "Both <code>joblib.load</code> and <code>pickle.load</code> can execute arbitrary code via the <code>__reduce__</code> protocol. "
    "A malicious 'model' file can drop a payload onto your server during the <code>load()</code> call. "
    "Treat <code>model.joblib</code> like an executable. Sign it, version-control it, hash-verify it. "
    "<b>Never <code>joblib.load('https://random-url.com/model.joblib')</code>.</b>")


### 5.5 — Forward reference: ONNX and FastAPI deployment (20.2)

In [58]:
box("industry", "Where this lands in production",
    "<b>For Python deployment</b> — Session 20.2 will write:<br>"
    "<pre>from fastapi import FastAPI\n"
    "import joblib, pandas as pd\n"
    "app = FastAPI()\n"
    "model = joblib.load('model.joblib')   # &larr; line one of the API\n"
    "@app.post('/predict')\n"
    "def predict(payload: dict):\n"
    "    return {'prediction': int(model.predict(pd.DataFrame([payload]))[0])}\n"
    "</pre>"
    "<b>For cross-language deployment</b> (Java, C#, Go, Rust) — convert to ONNX with <code>skl2onnx</code>. "
    "That's outside today's scope; the post-read links the path.")


---

## Section 6 — Mini-Project: Three Variants on the Same Test Set

Demonstrate the value of ColumnTransformer concretely.

### 6.1 — Variant A: Raw numeric features only (no scaling, drop categoricals)

In [59]:
pipe_A = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RNG)
pipe_A.fit(X_tr_h[num_cols_final], y_tr_h)
y_pred_A = pipe_A.predict(X_te_h[num_cols_final])
y_proba_A = pipe_A.predict_proba(X_te_h[num_cols_final])[:, 1]
f1_A = f1_score(y_te_h, y_pred_A)
ap_A = average_precision_score(y_te_h, y_proba_A)
auc_A = roc_auc_score(y_te_h, y_proba_A)
print(f"Variant A (raw numeric): F1={f1_A:.3f}, PR-AUC={ap_A:.3f}, ROC-AUC={auc_A:.3f}")


Variant A (raw numeric): F1=0.405, PR-AUC=0.428, ROC-AUC=0.779


### 6.2 — Variant B: StandardScaler on numerics only (still drops categoricals)

In [60]:
pipe_B = Pipeline([
    ("scaler", StandardScaler()),
    ("clf",    LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RNG)),
])
pipe_B.fit(X_tr_h[num_cols_final], y_tr_h)
y_pred_B = pipe_B.predict(X_te_h[num_cols_final])
y_proba_B = pipe_B.predict_proba(X_te_h[num_cols_final])[:, 1]
f1_B = f1_score(y_te_h, y_pred_B)
ap_B = average_precision_score(y_te_h, y_proba_B)
auc_B = roc_auc_score(y_te_h, y_proba_B)
print(f"Variant B (scaler only): F1={f1_B:.3f}, PR-AUC={ap_B:.3f}, ROC-AUC={auc_B:.3f}")


Variant B (scaler only): F1=0.409, PR-AUC=0.435, ROC-AUC=0.781


### 6.3 — Variant C: Full ColumnTransformer (numeric scaled + categorical OHE)

In [61]:
pipe_C = Pipeline([
    ("preprocess", ColumnTransformer([
        ("num", StandardScaler(),                                 num_cols_final),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols_final),
    ])),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RNG)),
])
pipe_C.fit(X_tr_h, y_tr_h)
y_pred_C = pipe_C.predict(X_te_h)
y_proba_C = pipe_C.predict_proba(X_te_h)[:, 1]
f1_C = f1_score(y_te_h, y_pred_C)
ap_C = average_precision_score(y_te_h, y_proba_C)
auc_C = roc_auc_score(y_te_h, y_proba_C)
print(f"Variant C (full ColumnTransformer): F1={f1_C:.3f}, PR-AUC={ap_C:.3f}, ROC-AUC={auc_C:.3f}")


Variant C (full ColumnTransformer): F1=0.418, PR-AUC=0.451, ROC-AUC=0.793


### 6.4 — Side-by-side comparison table

In [62]:
comp = pd.DataFrame({
    "Variant A — raw numeric":             {"F1": f1_A, "PR-AUC": ap_A, "ROC-AUC": auc_A},
    "Variant B — scaler only":             {"F1": f1_B, "PR-AUC": ap_B, "ROC-AUC": auc_B},
    "Variant C — full ColumnTransformer":  {"F1": f1_C, "PR-AUC": ap_C, "ROC-AUC": auc_C},
}).round(3).T
comp


,F1,PR-AUC,ROC-AUC
Variant A — raw numeric,0.405,0.428,0.779
Variant B — scaler only,0.409,0.435,0.781
Variant C — full ColumnTransformer,0.418,0.451,0.793


### 6.5 — The signature visualization (the money shot of this lab)

In [63]:
fig = go.Figure()
for col in ["F1", "PR-AUC", "ROC-AUC"]:
    fig.add_trace(go.Bar(
        x=comp.index, y=comp[col], name=col,
        text=[f"{v:.3f}" for v in comp[col]],
        textposition="outside",
        hovertemplate="%{x}<br>" + col + "=%{y:.3f}<extra></extra>",
    ))
fig.update_layout(
    barmode="group",
    title="The cost of skipping ColumnTransformer — Bank Marketing",
    template="plotly_white", height=440,
    yaxis=dict(range=[0, max(0.85, comp.values.max() + 0.1)]),
    xaxis_tickangle=-15,
)
fig.show()


In [64]:
lift_C_vs_B = f1_C - f1_B
lift_C_vs_A = f1_C - f1_A
box("output", "Reading the chart",
    f"Variant A (raw numeric) F1 = <b>{f1_A:.3f}</b>. "
    f"Variant B (scale numerics) F1 = <b>{f1_B:.3f}</b> — minimal change because we still threw away the categorical features. "
    f"Variant C (full ColumnTransformer) F1 = <b>{f1_C:.3f}</b> — a <b>+{lift_C_vs_B:.3f} F1 lift</b> over scaling-only, achieved purely by USING the categorical features at all. "
    "<b>This is the ColumnTransformer dividend.</b>")


In [65]:
# Hard assertion against the plan's verification bar
assert f1_C >= 0.40, f"Mini-project bar (F1 ≥ 0.40) not met. Got F1={f1_C:.3f}"
passed = "✓ PASSED" if f1_C >= 0.50 else "⚠ BELOW STRETCH TARGET"
box("output", "Final scorecard",
    f"Variant C: F1 = <b>{f1_C:.3f}</b>, PR-AUC = <b>{ap_C:.3f}</b>, ROC-AUC = <b>{auc_C:.3f}</b><br>"
    f"Status: <b>{passed}</b> (target: F1 ≥ 0.50; minimum: F1 ≥ 0.40).")


### 6.5b — Preprocessing Strategy Switcher (interactive)

**The headline interactive of this session.** The bar chart above shows three preprocessing variants on Bank Marketing. The chart below extends that to **five** strategies and lets you flip between them with a button click. Each panel updates: the metric bars, the confusion matrix, and the textual scorecard. Same train/test split for every strategy — only the preprocessing changes.

Notice the moment when adding categorical features (Raw \u2192 Numeric scaling \u2192 Full ColumnTransformer) gives the bigger jump than later refinements. Most of the win is in *using the data you have*; tuning comes after.


In [73]:
fig = make_subplots(rows=1, cols=2, column_widths=[0.55, 0.45],
    subplot_titles=('Metrics on test set', 'Confusion matrix on test set'),
    horizontal_spacing=0.18)

init_strategy = PREP_STRATEGIES[0]
init = PREP_CACHE[init_strategy]
metric_names = ['F1', 'Precision', 'Recall', 'PR-AUC', 'ROC-AUC']
init_vals = [init['F1'], init['Precision'], init['Recall'], init['AP'], init['ROC_AUC']]

fig.add_trace(go.Bar(x=metric_names, y=init_vals,
    text=[f'{v:.3f}' for v in init_vals], textposition='outside',
    marker=dict(color=MASAI_RED),
    hovertemplate='%{x}=%{y:.3f}<extra></extra>'), row=1, col=1)

tn0, fp0, fn0, tp0 = init['cm'].ravel()
cm_disp0 = [[fn0, tp0], [tn0, fp0]]
fig.add_trace(go.Heatmap(z=cm_disp0, x=['Pred 0', 'Pred 1'], y=['Actual 1', 'Actual 0'],
    text=cm_disp0, texttemplate='%{text}', textfont=dict(size=16),
    colorscale=[[0, '#fafafa'], [1, MASAI_RED]], showscale=False,
    hovertemplate='%{y} \u2192 %{x}<br>count=%{z}<extra></extra>'), row=1, col=2)

frames = []
for s in PREP_STRATEGIES:
    r = PREP_CACHE[s]
    vals = [r['F1'], r['Precision'], r['Recall'], r['AP'], r['ROC_AUC']]
    tn_, fp_, fn_, tp_ = r['cm'].ravel()
    cm_disp = [[fn_, tp_], [tn_, fp_]]
    frames.append(go.Frame(name=s, data=[
        go.Bar(x=metric_names, y=vals,
               text=[f'{v:.3f}' for v in vals], textposition='outside',
               marker=dict(color=MASAI_RED)),
        go.Heatmap(z=cm_disp, x=['Pred 0', 'Pred 1'], y=['Actual 1', 'Actual 0'],
                   text=cm_disp, texttemplate='%{text}', textfont=dict(size=16),
                   colorscale=[[0, '#fafafa'], [1, MASAI_RED]], showscale=False),
    ], layout=go.Layout(annotations=list(fig.layout.annotations) + [dict(
        text=f"<b>{s}</b><br>F1 = {r['F1']:.3f}<br>Precision = {r['Precision']:.3f}<br>Recall = {r['Recall']:.3f}<br>PR-AUC = {r['AP']:.3f}",
        xref='paper', yref='paper', x=0.99, y=-0.18, xanchor='right', yanchor='top',
        showarrow=False, align='left', font=dict(family='monospace', size=12),
        bgcolor='#fce4ec', bordercolor=MASAI_RED, borderwidth=1, borderpad=8)])))

fig.frames = frames
fig.update_layout(
    title='Preprocessing Strategy Switcher \u2014 click a button, watch the model change',
    template='plotly_white', height=500, showlegend=False,
    margin=dict(l=60, r=40, t=140, b=140),
    transition=dict(duration=600, easing='cubic-in-out'),
    annotations=list(fig.layout.annotations) + [dict(
        text=f"<b>{init_strategy}</b><br>F1 = {init['F1']:.3f}<br>Precision = {init['Precision']:.3f}<br>Recall = {init['Recall']:.3f}<br>PR-AUC = {init['AP']:.3f}",
        xref='paper', yref='paper', x=0.99, y=-0.18, xanchor='right', yanchor='top',
        showarrow=False, align='left', font=dict(family='monospace', size=12),
        bgcolor='#fce4ec', bordercolor=MASAI_RED, borderwidth=1, borderpad=8)],
    updatemenus=[dict(type='buttons', direction='right', showactive=True,
        x=0.5, y=1.22, xanchor='center', yanchor='top', pad=dict(t=5, b=5),
        buttons=[dict(label=s, method='animate',
            args=[[s], dict(mode='immediate',
                            frame=dict(duration=600, redraw=True),
                            transition=dict(duration=600, easing='cubic-in-out'))])
                 for s in PREP_STRATEGIES])],
)
fig.update_yaxes(range=[0, 1.05], row=1, col=1)
fig.show()


### 6.5c — Threshold slider on the full pipeline (interactive)

Variant C (full ColumnTransformer) is locked in. But the default 0.5 threshold is rarely optimal on imbalanced data — Bank Marketing is ~89:11. Drag the slider to find the threshold that maximizes F1, or that hits a recall floor your business team specifies.


In [74]:
from sklearn.metrics import precision_recall_curve as _prc
_proba_bm = PREP_CACHE['Full ColumnTransformer']['proba']
_p_bm, _r_bm, _ = _prc(y_te_pre, _proba_bm)
thr_steps_bm = np.round(np.arange(0.05, 0.96, 0.05), 2)

thr_bm_state = {}
for t in thr_steps_bm:
    y_pred_t = (_proba_bm >= t).astype(int)
    cm_t = confusion_matrix(y_te_pre, y_pred_t)
    p_t = precision_score(y_te_pre, y_pred_t, zero_division=0)
    r_t = recall_score(y_te_pre, y_pred_t)
    f1_t = f1_score(y_te_pre, y_pred_t, zero_division=0)
    thr_bm_state[float(t)] = dict(cm=cm_t, p=p_t, r=r_t, f1=f1_t)

fig = make_subplots(rows=1, cols=2, column_widths=[0.6, 0.4],
    subplot_titles=('PR curve (Bank Marketing) with movable marker', 'Confusion matrix at current threshold'),
    horizontal_spacing=0.18)
fig.add_trace(go.Scatter(x=_r_bm, y=_p_bm, mode='lines',
    line=dict(color=MASAI_RED, width=2),
    hovertemplate='Recall=%{x:.3f}<br>Precision=%{y:.3f}<extra></extra>'), row=1, col=1)
init_t = float(thr_steps_bm[len(thr_steps_bm)//2])
init_s = thr_bm_state[init_t]
fig.add_trace(go.Scatter(x=[init_s['r']], y=[init_s['p']], mode='markers',
    marker=dict(color='#1565C0', size=14, symbol='diamond'),
    hovertemplate=f't={init_t:.2f}<br>P={init_s["p"]:.2f}<br>R={init_s["r"]:.2f}<extra></extra>'), row=1, col=1)
tn0, fp0, fn0, tp0 = init_s['cm'].ravel()
cm_disp0 = [[fn0, tp0], [tn0, fp0]]
fig.add_trace(go.Heatmap(z=cm_disp0, x=['Pred 0', 'Pred 1'], y=['Actual 1', 'Actual 0'],
    text=cm_disp0, texttemplate='%{text}', textfont=dict(size=16),
    colorscale=[[0, '#fafafa'], [1, MASAI_RED]], showscale=False), row=1, col=2)

frames = []
for t in thr_steps_bm:
    s = thr_bm_state[float(t)]
    tn_, fp_, fn_, tp_ = s['cm'].ravel()
    cm_disp = [[fn_, tp_], [tn_, fp_]]
    frames.append(go.Frame(name=f'{t}', data=[
        go.Scatter(x=_r_bm, y=_p_bm, mode='lines', line=dict(color=MASAI_RED, width=2)),
        go.Scatter(x=[s['r']], y=[s['p']], mode='markers',
            marker=dict(color='#1565C0', size=14, symbol='diamond')),
        go.Heatmap(z=cm_disp, x=['Pred 0', 'Pred 1'], y=['Actual 1', 'Actual 0'],
            text=cm_disp, texttemplate='%{text}', textfont=dict(size=16),
            colorscale=[[0, '#fafafa'], [1, MASAI_RED]], showscale=False),
    ], layout=go.Layout(annotations=list(fig.layout.annotations) + [dict(
        text=f"<b>threshold = {t:.2f}</b><br>F1 = {s['f1']:.3f}<br>Precision = {s['p']:.3f}<br>Recall = {s['r']:.3f}",
        xref='paper', yref='paper', x=0.99, y=-0.18, xanchor='right', yanchor='top',
        showarrow=False, align='left', font=dict(family='monospace', size=12),
        bgcolor='#e3f2fd', bordercolor='#1565C0', borderwidth=1, borderpad=8)])))

fig.frames = frames
fig.update_layout(
    title='Threshold slider \u2014 Bank Marketing pipeline',
    template='plotly_white', height=460, showlegend=False,
    margin=dict(l=60, r=40, t=80, b=140),
    transition=dict(duration=400, easing='cubic-in-out'),
    annotations=list(fig.layout.annotations) + [dict(
        text=f"<b>threshold = {init_t:.2f}</b><br>F1 = {init_s['f1']:.3f}<br>Precision = {init_s['p']:.3f}<br>Recall = {init_s['r']:.3f}",
        xref='paper', yref='paper', x=0.99, y=-0.18, xanchor='right', yanchor='top',
        showarrow=False, align='left', font=dict(family='monospace', size=12),
        bgcolor='#e3f2fd', bordercolor='#1565C0', borderwidth=1, borderpad=8)],
    sliders=[dict(active=len(thr_steps_bm)//2, x=0.05, y=-0.05, len=0.9,
        pad=dict(t=20, b=10),
        currentvalue=dict(prefix='threshold = ', font=dict(size=14, color='#1565C0')),
        steps=[dict(method='animate', label=f'{t:.2f}',
            args=[[f'{t}'], dict(mode='immediate',
                                  frame=dict(duration=400, redraw=True),
                                  transition=dict(duration=400, easing='cubic-in-out'))])
               for t in thr_steps_bm])],
)
fig.update_xaxes(title_text='Recall', range=[0, 1], row=1, col=1)
fig.update_yaxes(title_text='Precision', range=[0, 1.05], row=1, col=1)
fig.show()


### 6.5d — Decision flowchart: which encoder for which column?

Quick visual reference for picking the right encoder given a column's type and cardinality.


In [75]:
nodes = [
    ('root', 0.50, 1.00, 'New column?', 'Start here every time you add a column.', '#1565C0'),
    ('typ',  0.50, 0.78, 'Numeric or<br>Categorical?', 'Check df.dtypes', '#888'),
    ('num',  0.18, 0.55, 'Numeric', 'Continuous or count column.', '#888'),
    ('cat',  0.82, 0.55, 'Categorical', 'object dtype or low-cardinality int.', '#888'),
    ('numA', 0.05, 0.30, 'StandardScaler<br>(default)', 'Mean-zero, unit-variance.', '#2E7D32'),
    ('numB', 0.32, 0.30, 'np.log1p<br>(if right-skewed)', 'For lognormal-ish features.', '#2E7D32'),
    ('lo',   0.65, 0.30, 'Low cardinality<br>(< 30)', '', '#888'),
    ('hi',   0.92, 0.30, 'High cardinality<br>(>= 30)', '', '#888'),
    ('ohe',  0.65, 0.05, 'OneHotEncoder<br>(handle_unknown=ignore)', 'Production-safe default.', '#E65100'),
    ('freq', 0.92, 0.05, 'Frequency encode<br>or TargetEncoder', 'Avoid OHE blow-up.', '#E65100'),
]
edges = [('root','typ'),('typ','num'),('typ','cat'),('num','numA'),('num','numB'),('cat','lo'),('cat','hi'),('lo','ohe'),('hi','freq')]
node_pos = {nid: (x, y) for nid, x, y, *_ in nodes}

fig = go.Figure()
for src, dst in edges:
    x0, y0 = node_pos[src]; x1, y1 = node_pos[dst]
    fig.add_trace(go.Scatter(x=[x0, x1], y=[y0, y1], mode='lines',
        line=dict(color='#bbb', width=1.5), hoverinfo='skip', showlegend=False))
for nid, x, y, label, hover, color in nodes:
    fig.add_trace(go.Scatter(x=[x], y=[y], mode='markers+text',
        marker=dict(size=52, color=color, line=dict(color='white', width=2)),
        text=[f'<b>{label}</b>'], textposition='middle center', textfont=dict(size=10, color='white'),
        hovertext=hover, hoverinfo='text', showlegend=False))
fig.update_layout(
    title='Encoder decision flowchart \u2014 hover for the recommendation',
    template='plotly_white', height=520,
    margin=dict(l=20, r=20, t=60, b=20),
    xaxis=dict(visible=False, range=[-0.05, 1.05]),
    yaxis=dict(visible=False, range=[-0.05, 1.10]),
)
fig.show()


### 6.6 — Production deployment checklist (the take-home artifact)

```
PRODUCTION DEPLOYMENT CHECKLIST
────────────────────────────────────────────────────────────────
  ☐ 1.  Pipeline includes ALL preprocessing (no manual transform calls)
  ☐ 2.  OneHotEncoder uses handle_unknown='ignore'
  ☐ 3.  Stratified train/test split with explicit random_state
  ☐ 4.  Cross-validation uses StratifiedKFold + scoring='average_precision'
  ☐ 5.  joblib.dump with compress=3; file size logged
  ☐ 6.  Round-trip test: load in fresh process, predict on test row
  ☐ 7.  requirements.txt pins scikit-learn, plotly, joblib versions
  ☐ 8.  Model.joblib is treated as an executable (signed, hash-verified)
```


### 6.7 — 10 Key Takeaways

1. **Feature engineering is the highest-leverage decision in ML.** Better features beat better models.
2. **Always ask: "Could this feature exist before the prediction time?"** If no, drop it. The Bank Marketing `duration` is a textbook leakage trap.
3. **Sentinel values (`pdays=999`) silently corrupt scaling.** Split into a boolean flag + clean numeric.
4. **`OneHotEncoder(handle_unknown='ignore')` is mandatory in production.** The default raises on unseen categories.
5. **`ColumnTransformer` is the only correct way to preprocess mixed types.** Manual `pd.get_dummies + StandardScaler` has three production-killing bugs in five lines.
6. **`Pipeline([(preprocess, CT), (clf, model)])` is the production unit.** One `.fit()`, one `.predict()`.
7. **`cross_val_score(pipe, ...)` is leakage-free by construction.** Each fold's preprocessor is fit only on that fold's training portion.
8. **`PolynomialFeatures(interaction_only=True)` produces `n + n(n-1)/2` outputs.** Don't blindly polynomial-expand wide datasets.
9. **`joblib.dump(pipe, 'model.joblib', compress=3)` saves the whole Pipeline.** `joblib.load` returns a `predict()`-ready object.
10. **`joblib.load` and `pickle.load` execute arbitrary code.** Treat `model.joblib` like an executable.


### 6.8 — What's Next

- **13.1 (Git, Version Control & Code Quality)** — `model.joblib` + `requirements.txt` + `README.md` go into a Git repo. Today's pipeline becomes a versioned artifact.
- **13.2 (Regularization — Ridge/Lasso/ElasticNet)** — slot regularized estimators into the Pipeline's `clf` step. Lasso on the OHE-expanded matrix is a real feature-selection demo.
- **14.1 (Advanced Regression)** — VIF on the polynomial-expanded matrix; multicollinearity diagnosis.
- **14.2 (Time Series)** — aggregation features (rolling means, lag features).
- **15.x (Trees / Random Forests)** — same Pipeline pattern, drop the StandardScaler step.
- **16.1 (XGBoost / LightGBM / CatBoost)** — native categorical handling vs `OneHotEncoder`.
- **16.2 (Hyperparameter Tuning)** — `GridSearchCV(pipe, {'clf__C': ..., 'preprocess__num__with_mean': ...})` uses today's namespaced parameter syntax.
- **20.2 (FastAPI & Docker)** — `joblib.load('model.joblib')` is line one of the inference endpoint.
- **21.1 (MLOps — CI/CD & Drift Detection)** — Pipeline + requirements.txt + model.joblib is the deployment unit.


### 6.9 — Common Pitfalls Quick Reference

| Symptom | Root cause | Fix |
|---|---|---|
| `ValueError: Found unknown categories ['Z']` in production | `OneHotEncoder` default raises on unseen categories | `OneHotEncoder(handle_unknown='ignore')` |
| `TypeError: ... unexpected keyword argument 'sparse'` | sklearn 1.2+ renamed `sparse` to `sparse_output` | Use `sparse_output=False` |
| `pd.read_csv` returns one column instead of 21 | Bank Marketing uses `;` separator | Pass `sep=';'` |
| F1 = 0.87 in CV, 0.40 in production | Used a post-event feature (`duration`) | Drop leaky features; ask the diagnostic question |
| `get_feature_names_out()` raises `NotFittedError` | Called on a Pipeline not yet fitted | `pipe.fit(X_train, y_train)` first |
| `joblib.load` returns a Pipeline, but `.predict()` shape error | sklearn version mismatch | Pin sklearn in `requirements.txt`; consider ONNX |
| Memory blow-up with `PolynomialFeatures(degree=3)` | Cubic combinatorial expansion | Stick to `degree=2, interaction_only=True`; let Lasso (13.2) prune |

---

**End of Practical Lab — Session 12.2.**

*Vishlesan i-Hub IIT Patna × Masai School*
